# bench_fin — Next-Day Return Classification with Advanced Tabular Augmentation & SSL

This notebook refreshes the financial benchmark to include
- modern oversampling strategies (simplicial SMOTE, MEB-SMOTE, and an
  MGS-GRF-inspired grouped sampler), and
- semi-supervised fine-tuning using the shared tabular SSL helpers.

We train on the [S&P 500 Stocks](https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks)
collection and evaluate on the
[World Stock Prices (Daily Updating)](https://www.kaggle.com/datasets/nelgiriyewithana/world-stock-prices-daily-updating)
records.


## 1. Imports & configuration
We reuse `BenchmarkRunner`, the tabular registry, and the semi-supervised wrapper
from `pipelines_torch`. Augmentations are implemented inline to keep the notebook
self-contained.


In [ ]:
import os
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import TensorDataset, DataLoader

from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.models import MODEL_REGISTRY
from pipelines_torch.ss_models import SemiSupervisedTabular
from pipelines_torch.base import SimplePredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
print(f"Using device: {DEVICE}")


In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


### Download datasets (optional Kaggle helper)
The helper populates `data_fin/sp500` and `data_fin/world` if they are missing.
Skip this cell if you already have the CSV files locally.


In [ ]:
DATA_ROOT = Path("data_fin")
SP500_DIR = DATA_ROOT / "sp500"
WORLD_DIR = DATA_ROOT / "world"
SP500_SLUG = "andrewmvd/sp-500-stocks"
WORLD_SLUG = "nelgiriyewithana/world-stock-prices-daily-updating"

for directory in (SP500_DIR, WORLD_DIR):
    directory.mkdir(parents=True, exist_ok=True)

try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    if not any(SP500_DIR.iterdir()):
        print(f"Downloading {SP500_SLUG} …")
        api.dataset_download_files(SP500_SLUG, path=SP500_DIR, unzip=True)
    if not any(WORLD_DIR.iterdir()):
        print(f"Downloading {WORLD_SLUG} …")
        api.dataset_download_files(WORLD_SLUG, path=WORLD_DIR, unzip=True)
except Exception as exc:  # pragma: no cover - runtime dependent
    print(f"Kaggle download skipped or failed: {exc}")


## 2. Data loading & feature engineering
The loader scans each directory for price tables (containing `Date` and `Close`).
If a metadata file with `Symbol`/`Sector` exists it is merged automatically.


In [ ]:
def _normalise_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    for col in df.columns:
        lower = col.lower()
        if lower in {"ticker", "symbol", "code", "name"}:
            rename_map[col] = "Symbol"
        elif lower == "date":
            rename_map[col] = "Date"
        elif lower == "closeprice":
            rename_map[col] = "Close"
        elif lower == "openprice":
            rename_map[col] = "Open"
        elif lower == "highprice":
            rename_map[col] = "High"
        elif lower == "lowprice":
            rename_map[col] = "Low"
    return df.rename(columns=rename_map)


def load_market_dataset(root: Path) -> pd.DataFrame:
    frames = []
    sector_info = None
    for path in root.rglob("*.csv"):
        df = pd.read_csv(path)
        df = _normalise_columns(df)
        cols = set(df.columns.str.lower())
        if {"symbol", "sector"}.issubset(df.columns):
            # Potential metadata file
            if "date" not in df.columns and "close" not in df.columns:
                sector_info = df[["Symbol", "Sector"]].drop_duplicates()
                continue
        if {"date", "close"}.issubset(df.columns):
            if "Symbol" not in df.columns:
                df["Symbol"] = path.stem
            frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No price tables found under {root}")
    data = pd.concat(frames, ignore_index=True)
    data["Date"] = pd.to_datetime(data["Date"])
    if sector_info is not None:
        data = data.merge(sector_info, on="Symbol", how="left")
    if "Sector" not in data.columns:
        data["Sector"] = "Unknown"
    return data


In [ ]:
def engineer_features(df: pd.DataFrame, *, horizon: int = 1) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["Symbol", "Date"])
    df["return"] = df.groupby("Symbol")["Close"].pct_change()
    df["return_next"] = df.groupby("Symbol")["Close"].pct_change().shift(-horizon)
    df["hlv"] = (df["High"] - df["Low"]) / df["Close"]
    for window in (3, 5, 10):
        df[f"ma_{window}"] = df.groupby("Symbol")["Close"].transform(lambda s: s.rolling(window).mean())
        df[f"vol_{window}"] = df.groupby("Symbol")["return"].transform(lambda s: s.rolling(window).std())
    df = df.dropna().reset_index(drop=True)
    df["sector_code"] = df["Sector"].astype("category").cat.codes
    df["y_binary"] = (df["return_next"] > 0).astype(int)
    q_low, q_high = df["return_next"].quantile([0.33, 0.66])
    df["y_multiclass"] = np.select(
        [df["return_next"] <= q_low, df["return_next"] >= q_high],
        [0, 2],
        default=1,
    )
    return df


In [ ]:
sp500_prices = load_market_dataset(SP500_DIR)
world_prices = load_market_dataset(WORLD_DIR)
train_df = engineer_features(sp500_prices)
world_df = engineer_features(world_prices)
print(f"Training rows: {len(train_df)}, Evaluation rows: {len(world_df)}")


In [ ]:
feature_cols = [
    "sector_code",
    "return",
    "hlv",
    "ma_3",
    "ma_5",
    "ma_10",
    "vol_3",
    "vol_5",
    "vol_10",
]
continuous_cols = feature_cols[1:]
scaler = StandardScaler()
train_scaled = train_df.copy()
train_scaled[continuous_cols] = scaler.fit_transform(train_scaled[continuous_cols])
world_scaled = world_df.copy()
world_scaled[continuous_cols] = scaler.transform(world_scaled[continuous_cols])

X = train_scaled[feature_cols].to_numpy(dtype=np.float32)
y_binary = train_scaled["y_binary"].to_numpy(dtype=np.int64)
y_multi = train_scaled["y_multiclass"].to_numpy(dtype=np.int64)
world_X = world_scaled[feature_cols].to_numpy(dtype=np.float32)
world_y_binary = world_scaled["y_binary"].to_numpy(dtype=np.int64)
world_y_multi = world_scaled["y_multiclass"].to_numpy(dtype=np.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_binary, test_size=0.2, random_state=SEED, stratify=y_binary
)
print(f"Feature matrix: {X.shape}, Validation split: {X_val.shape}")


## 3. Augmentation primitives (simplicial SMOTE, MEB-SMOTE, MGS-GRF style)
The helpers operate on the continuous features (excluding the sector code in the
first column) and then reattach the group identifier.


In [ ]:
from sklearn.neighbors import NearestNeighbors


def simplicial_smote(features: np.ndarray, labels: np.ndarray, minority: int, n_samples: int, rng: np.random.Generator) -> np.ndarray:
    X_min = features[labels == minority]
    if len(X_min) < 2 or n_samples <= 0:
        return np.empty((0, features.shape[1]))
    n_neighbors = min(6, len(X_min))
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(X_min)
    synthetic = []
    for _ in range(n_samples):
        idx = rng.integers(0, len(X_min))
        distances, indices = nbrs.kneighbors(X_min[[idx]])
        candidates = X_min[indices[0][1:]]
        if len(candidates) == 0:
            continue
        r = rng.integers(1, len(candidates) + 1)
        sample = candidates[rng.choice(len(candidates), size=r, replace=False)]
        weights = rng.random(r)
        weights /= weights.sum()
        synthetic.append(np.sum(sample * weights[:, None], axis=0))
    return np.asarray(synthetic)


def meb_smote(features: np.ndarray, labels: np.ndarray, minority: int, n_samples: int, rng: np.random.Generator) -> np.ndarray:
    X_min = features[labels == minority]
    if len(X_min) == 0 or n_samples <= 0:
        return np.empty((0, features.shape[1]))
    center = X_min.mean(axis=0)
    radius = np.linalg.norm(X_min - center, axis=1).max()
    noise = rng.uniform(-1, 1, size=(n_samples, features.shape[1])) * (radius * 0.2)
    return center + noise


def mgs_grf_like(
    features: np.ndarray,
    labels: np.ndarray,
    groups: np.ndarray,
    minority: int,
    n_samples: int,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    if n_samples <= 0:
        return np.empty((0, features.shape[1])), np.empty((0,), dtype=groups.dtype)
    group_values = np.unique(groups[labels == minority])
    synthetic = []
    synthetic_groups = []
    per_group = max(1, n_samples // max(1, len(group_values)))
    for g in group_values:
        subset = features[(labels == minority) & (groups == g)]
        if len(subset) < 3:
            continue
        cov = np.cov(subset, rowvar=False)
        cov += np.eye(cov.shape[0]) * 1e-3
        samples = rng.multivariate_normal(subset.mean(axis=0), cov, size=per_group)
        synthetic.append(samples)
        synthetic_groups.append(np.full(per_group, g, dtype=groups.dtype))
    if not synthetic:
        return np.empty((0, features.shape[1])), np.empty((0,), dtype=groups.dtype)
    stacked = np.vstack(synthetic)
    stacked_groups = np.concatenate(synthetic_groups)
    return stacked[:n_samples], stacked_groups[:n_samples]


In [ ]:
def financial_augmenter(X: np.ndarray, y: np.ndarray, *, max_factor: float = 2.0, random_state: int = 42):
    rng = np.random.default_rng(random_state)
    groups = X[:, 0].astype(int)
    features = X[:, 1:]
    minority = 1 if (y == 1).sum() < (y == 0).sum() else 0
    minority_count = (y == minority).sum()
    majority_count = len(y) - minority_count
    target_minority = int(max(majority_count / max_factor, minority_count))
    needed = max(0, target_minority - minority_count)
    if needed == 0:
        return X, y
    share = max(1, needed // 3)
    simp = simplicial_smote(features, y, minority, share, rng)
    meb = meb_smote(features, y, minority, share, rng)
    remaining = needed - len(simp) - len(meb)
    mgs_samples, mgs_groups = mgs_grf_like(features, y, groups, minority, remaining, rng)

    def sample_groups(count: int) -> np.ndarray:
        minority_groups = groups[y == minority]
        if len(minority_groups) == 0:
            return np.zeros(count, dtype=groups.dtype)
        return minority_groups[rng.integers(0, len(minority_groups), size=count)]

    new_features = []
    new_groups = []
    if len(simp):
        new_features.append(simp)
        new_groups.append(sample_groups(len(simp)))
    if len(meb):
        new_features.append(meb)
        new_groups.append(sample_groups(len(meb)))
    if len(mgs_samples):
        new_features.append(mgs_samples)
        new_groups.append(mgs_groups)

    if not new_features:
        return X, y

    stacked_features = np.vstack(new_features)
    stacked_groups = np.concatenate(new_groups)
    augmented_features = np.hstack([stacked_groups[:, None].astype(X.dtype), stacked_features.astype(X.dtype)])
    augmented_labels = np.full(len(augmented_features), minority, dtype=y.dtype)

    X_aug = np.vstack([X, augmented_features])
    y_aug = np.concatenate([y, augmented_labels])
    return X_aug, y_aug


## 4. Benchmark with augmentation grid
We compare runs with/without augmentation using the registry MLPs.


In [ ]:
def accuracy_metric(y_true, probs):
    preds = np.argmax(probs, axis=1)
    return accuracy_score(y_true, preds)


def f1_macro_metric(y_true, probs):
    preds = np.argmax(probs, axis=1)
    return f1_score(y_true, preds, average="macro")


def roc_auc_metric(y_true, probs):
    try:
        return roc_auc_score(y_true, probs[:, 1])
    except Exception:
        return float("nan")

accuracy_metric.__name__ = "accuracy"
f1_macro_metric.__name__ = "f1_macro"
roc_auc_metric.__name__ = "roc_auc"
metrics = [accuracy_metric, f1_macro_metric, roc_auc_metric]


In [ ]:
input_dim = X_train.shape[1]
model_configs = [
    {
        "name": "mlp_classifier",
        "class": MODEL_REGISTRY["mlp_classifier"],
        "params": {
            "input_dim": input_dim,
            "hidden_dims": [512, 256, 128],
            "num_classes": 2,
            "dropout": 0.3,
            "batchnorm": True,
        },
    },
    {
        "name": "deep_mlp_classifier",
        "class": MODEL_REGISTRY["deep_mlp_classifier"],
        "params": {
            "input_dim": input_dim,
            "num_classes": 2,
        },
    },
]
augmentations = [None, financial_augmenter]


In [ ]:
runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=augmentations,
    metrics=metrics,
    task_type="classification",
    device=DEVICE,
    epochs=12,
    batch_size=128,
    learning_rate=3e-4,
    weight_decay=1e-4,
    use_class_weights=True,
    use_kfold=False,
    random_state=SEED,
    path_start="bench_fin_tabular",
    max_factor=2.0,
)
results_tabular = runner.run(X_train, y_train)
results_tabular.sort_values("score", ascending=False)


### Evaluate on validation and world indices
We reload the saved checkpoints to score both the held-out validation split and
the external world dataset.


In [ ]:
def evaluate_fin_models(model_names: List[str], augmentation_names: List[str], X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    rows = []
    for model_name in model_names:
        params = next(cfg["params"] for cfg in model_configs if cfg["name"] == model_name)
        for aug in augmentation_names:
            ckpt = f"{model_name}_{'none' if aug is None else 'financial_augmenter'}"
            ckpt_path = Path("results") / "bench_fin_tabular" / f"{ckpt}.pt"
            if not ckpt_path.exists():
                continue
            model = MODEL_REGISTRY[model_name](**params)
            state_dict = torch.load(ckpt_path, map_location=DEVICE)
            model.load_state_dict(state_dict)
            predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=256)
            probs = predictor.predict_proba(X)
            rows.append({
                "model": model_name,
                "augmentation": "none" if aug is None else "financial_augmenter",
                "accuracy": accuracy_metric(y, probs),
                "f1_macro": f1_macro_metric(y, probs),
                "roc_auc": roc_auc_metric(y, probs),
            })
    return pd.DataFrame(rows)

val_eval = evaluate_fin_models([cfg["name"] for cfg in model_configs], augmentations, X_val, y_val)
val_eval


In [ ]:
world_eval = evaluate_fin_models([cfg["name"] for cfg in model_configs], augmentations, world_X, world_y_binary)
world_eval


## 5. Semi-supervised fine-tuning
We reuse `SemiSupervisedTabular` with MeanTeacher to incorporate unlabeled rows.


In [ ]:
def train_fin_ssl(
    X: np.ndarray,
    y: np.ndarray,
    *,
    unlabeled_fraction: float = 0.3,
    epochs: int = 20,
    batch_size: int = 256,
    lr: float = 3e-4,
) -> Tuple[SemiSupervisedTabular, pd.DataFrame]:
    base_model = MODEL_REGISTRY["mlp_classifier"](
        input_dim=X.shape[1],
        hidden_dims=[512, 256, 128],
        num_classes=2,
        dropout=0.3,
        batchnorm=True,
    )
    ssl_model = SemiSupervisedTabular(base_model, num_classes=2, use_mean_teacher=True).to(DEVICE)
    optimizer = torch.optim.AdamW(ssl_model.parameters(), lr=lr, weight_decay=1e-4)

    rng = np.random.default_rng(SEED)
    mask = rng.random(len(y)) > unlabeled_fraction
    labeled_idx = np.where(mask)[0]
    unlabeled_idx = np.where(~mask)[0]

    labeled_ds = TensorDataset(
        torch.tensor(X[labeled_idx], dtype=torch.float32),
        torch.tensor(y[labeled_idx], dtype=torch.long),
    )
    unlabeled_ds = TensorDataset(
        torch.tensor(X[unlabeled_idx], dtype=torch.float32),
        torch.zeros(len(unlabeled_idx)),
    )

    labeled_loader = DataLoader(labeled_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    unlabeled_loader = DataLoader(unlabeled_ds, batch_size=batch_size * 2, shuffle=True, drop_last=True)

    history = []
    from itertools import cycle

    unlabeled_iter = cycle(unlabeled_loader) if len(unlabeled_ds) > 0 else None

    for epoch in range(epochs):
        if unlabeled_iter is None:
            break
        ssl_model.train()
        total_loss = 0.0
        steps = 0
        for xb_l, yb_l in labeled_loader:
            xb_u, _ = next(unlabeled_iter)
            xb_l = xb_l.to(DEVICE)
            yb_l = yb_l.to(DEVICE)
            xb_u = xb_u.to(DEVICE)

            optimizer.zero_grad()
            loss, logs = ssl_model.step((xb_l, yb_l), (xb_u, None), epoch)
            loss.backward()
            optimizer.step()
            ssl_model.post_step()

            total_loss += loss.item()
            steps += 1

        ssl_model.eval()
        with torch.no_grad():
            logits = ssl_model(torch.tensor(X_val, dtype=torch.float32, device=DEVICE))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            val_acc = accuracy_score(y_val, preds)
            val_f1 = f1_score(y_val, preds, average="macro")
        history.append({"epoch": epoch + 1, "loss": total_loss / max(1, steps), "val_accuracy": val_acc, "val_f1": val_f1})
        print(f"Epoch {epoch+1:02d}: loss={history[-1]['loss']:.4f} val_f1={val_f1:.3f}")

    return ssl_model, pd.DataFrame(history)


In [ ]:
ssl_model, ssl_history = train_fin_ssl(X, y_binary, epochs=15, unlabeled_fraction=0.35)
ssl_history


In [ ]:
def evaluate_ssl_model(model: torch.nn.Module, X: np.ndarray, y: np.ndarray) -> pd.Series:
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32, device=DEVICE))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = np.argmax(probs, axis=1)
    return pd.Series({
        "accuracy": accuracy_score(y, preds),
        "f1_macro": f1_score(y, preds, average="macro"),
        "roc_auc": roc_auc_metric(y, probs),
    })

ssl_val_metrics = evaluate_ssl_model(ssl_model, X_val, y_val)
print("SSL validation metrics:
", ssl_val_metrics)
world_ssl_metrics = evaluate_ssl_model(ssl_model, world_X, world_y_binary)
print("
SSL world metrics:
", world_ssl_metrics)


### Multi-class extension
To evaluate the three-class problem, simply swap `y_binary` with `y_multi` in the
cells above and adjust the metrics (e.g. macro-averaged F1 only). The augmentation
function remains valid because it operates on the minority class per run.
